In [1]:
import torch
import re
import gc 

import pandas as pd
import numpy as np

from collections import defaultdict
from tqdm.notebook import tqdm
tqdm.pandas()

from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
triples = pd.read_excel("../outputs/raw_outputs/generated_triples_coalesced.xlsx").drop("Unnamed: 0", axis=1)
triples.head()

,model,prompt,id,text,triples
0,gemma,structured,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('Job_Title', 'HAS_LOCATION', 'London, UK'), ..."
1,gemma,structured,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('IT-administrator', 'OFFERS_POSITION', 'ALD ..."
2,gemma,structured,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mine...,"[('Brande', 'HAS_LOCATION', 'Denmark'), ('Aqua..."
3,gemma,semi-structured,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...,"[('Job title', 'requires_skill', 'IT operation..."
4,gemma,semi-structured,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,...","[('Job', 'has_title', 'IT-administrator'), ('J..."


### Prepare ISCO data

In [3]:
def prepare_iscos(row):
    row["Included occupations"] = row["Included occupations"].replace('Examples of the occupations classified here:', '')
    row["Included occupations"] = "".join(row["Included occupations"].split("\n")[:4]).replace("- ", ", ")

    return {row['ISCO 08 Code']: f"{row['Title EN']}. {row['Definition']} (e.g.{row['Included occupations']})"}

In [4]:
iscos = pd.read_excel("ISCO-08.xlsx")
iscos = iscos[(iscos["ISCO 08 Code"].astype(str).str.fullmatch(r'\d{4}'))][["ISCO 08 Code", "Title EN", 
                                                                            "Included occupations", "Definition"]]
# TODO: Ugly, clean
result = iscos.apply(lambda row: prepare_iscos(row), axis=1).values

r_isco = {}

for d in result:
    r_isco.update(d)

### Embed ISCOs and triples

In [5]:
emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

isco_embs = {}

# Compute embedding for both lists
for isco, text in r_isco.items():
    isco_embs[isco] = emb_model.encode(text, convert_to_tensor=True)    

In [6]:
isco_ids = list(isco_embs.keys())
isco_matrix = torch.stack([isco_embs[_id] for _id in isco_ids]).to("cuda:0")  # (N, d)

# Normalize once for cosine similarity
isco_matrix_norm = isco_matrix / isco_matrix.norm(dim=1, keepdim=True)

In [7]:
def find_matches(query_embedding, ids, matrix_norm, k=20):
    """
    query_embedding: torch.Tensor on cuda, shape [d]
    isco_ids: list of IDs (pre-built)
    isco_matrix_norm: pre-normalized matrix (N, d)
    k: number of matches to return
    """
    # Normalize query
    q = query_embedding / query_embedding.norm(dim=0, keepdim=True)

    # Compute cosine similarity (N)
    sims = torch.matmul(matrix_norm, q)

    # Top-k
    topk_vals, topk_idx = torch.topk(sims, k)

    return [ids[i] for i in topk_idx.tolist()]


In [8]:
triples["embedding"] = triples["triples"].progress_apply(
    lambda x: emb_model.encode(x, convert_to_tensor=True).to("cuda:0")
)

triples["Top matches isco"] = triples["embedding"].progress_apply(
    lambda emb: find_matches(emb, isco_ids, isco_matrix_norm, k=20)
)

  0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

### Embed ESCOs

In [9]:
df_skills = pd.read_csv("skills_en.csv")
df_skills["id"] = df_skills.index
df_skills = df_skills[["id", "skillType", "preferredLabel", "description"]]

df_skills.head()

,id,skillType,preferredLabel,description
0,0,skill/competence,manage musical staff,Assign and manage staff tasks in areas such as...
1,1,skill/competence,supervise correctional procedures,Supervise the operations of a correctional fac...
2,2,skill/competence,apply anti-oppressive practices,"Identify oppression in societies, economies, c..."
3,3,skill/competence,control compliance of railway vehicles regulat...,"Inspect rolling stock, components and systems ..."
4,4,skill/competence,identify available services,Identify the different services available for ...


In [10]:
df_skills.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13939 entries, 0 to 13938
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              13939 non-null  int64 
 1   skillType       13934 non-null  object
 2   preferredLabel  13939 non-null  object
 3   description     13939 non-null  object
dtypes: int64(1), object(3)
memory usage: 435.7+ KB


In [11]:
def prepare_escos(row):
    return {f"{row['id']:06d}" : f"{row['preferredLabel']}: {row['description']}"}

# TODO: Ugly, clean
result = df_skills.apply(lambda row: prepare_escos(row), axis=1).values

r_esco = {}

for d in result:
    r_esco.update(d)

In [13]:
embs_esco = {}

# Compute embedding for both lists
for esco, text in tqdm(r_esco.items()):
    embs_esco[esco] = emb_model.encode(text, convert_to_tensor=True)   

  0%|          | 0/13939 [00:00<?, ?it/s]

In [14]:
esco_ids = list(embs_esco.keys())
esco_matrix = torch.stack([embs_esco[_id] for _id in esco_ids]).to("cuda:0")  # (N, d)

# Normalize once for cosine similarity
esco_matrix_norm = esco_matrix / esco_matrix.norm(dim=1, keepdim=True)

triples["Top matches esco"] = triples["embedding"].progress_apply(
    lambda emb: find_matches(emb, esco_ids, esco_matrix_norm, k=20)
)

  0%|          | 0/27 [00:00<?, ?it/s]

### Link ISCO/ESCO to triples

In [15]:
def run_pipeline(model, tokenizer, r, row, task="isco"):
    
    text = row["text"]

    if task == "isco":
        matches = [r[_id] for _id in row["Top matches isco"]]
        
        prompt = f"""
        Link the subjects and objects in these triples to their respective ISCO-08 codes (where applicable).
        A link takes the shape: (*subject/object*, has_isco, ISCO_XXXX) with XXXX replaced by the appropriate 4-digit code.
        Ensure that it matches this shape exactly. 
        
        Pick from the following definitions:
        {matches}
        
        return the found links as a list of triples. Do not add any text to your output other than the subjects, 
        predicates, and objects. 
        
        Here are the triples: 
        """
    elif task == "esco":
        matches = [r[_id] for _id in row["Top matches esco"]]
        
        prompt = f"""
        Link the subjects and objects in these triples to their respective ESCO skill codes (where applicable).
        A link takes the shape: (*skill/knowledge*, has_esco, ESCO_*code*) with *skill/knowledge* being replace by the 
        appropriate node and *code* replaced by the appropriate code.
        Ensure that it matches this shape exactly. The majority of the text will be in Danish. 
        
        Use the following definitions:
        {matches}
        
        return the found links as a list of triples. Do not add any text to your output other than the subjects, predicates, and objects.
        """

    if not text:
        return []

    # Create message
    messages = [
        {"role": "user", "content": prompt + text}
    ]

    # Apply template
    processed_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize
    model_inputs = tokenizer([processed_text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4096 # 16384
    )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # Return response
    return tokenizer.decode(output_ids, skip_special_tokens=True)

In [15]:
models = {"qwen" : "Qwen/Qwen3-4B-Instruct-2507",
          "llama" : "meta-llama/Llama-3.2-3B-Instruct",
          "gemma" : "google/gemma-3n-e4b-it"}

device = ("cuda:0" if torch.cuda.is_available() else "cpu")

final_results = {}

for model_name, hf in models.items():   
    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(hf)
    model = AutoModelForCausalLM.from_pretrained(
        hf,
        dtype="auto",
        device_map=None
    ).to(device)

    # Filter for triples generated by this model
    curr_triples = triples[triples["model"] == model_name]
    
    curr_triples["ISCO"] = curr_triples.progress_apply(lambda x: run_pipeline(model, tokenizer, r_isco, x, task="isco"), axis=1)
    curr_triples["ESCO"] = curr_triples.progress_apply(lambda x: run_pipeline(model, tokenizer, r_esco, x, task="esco"), axis=1)

    final_results[model_name] = curr_triples

    del model
    del tokenizer
    torch.cuda.empty_cache() 
    torch.cuda.ipc_collect()
    gc.collect()

triples_ISCO_ESCO = pd.concat(final_results.values(), ignore_index=True)
triples_ISCO_ESCO[["model", "prompt", "id", "text",
                   "triples", "ISCO", "ESCO"]].to_excel("../outputs/raw_outputs/triples_with_ISCO_ESCO.xlsx")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

C:\Users\roans\AppData\Local\Temp\ipykernel_3136\1147770738.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_triples["ISCO"] = curr_triples.progress_apply(lambda x: run_pipeline(model, tokenizer, r_isco, x, task="isco"), axis=1)


  0%|          | 0/9 [00:00<?, ?it/s]

C:\Users\roans\AppData\Local\Temp\ipykernel_3136\1147770738.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  curr_triples["ESCO"] = curr_triples.progress_apply(lambda x: run_pipeline(model, tokenizer, r_esco, x, task="esco"), axis=1)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
C:\Users\roans\AppData\Local\Temp\ipykernel_3136\1147770738.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-ve

  0%|          | 0/9 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.

KeyboardInterrupt


KeyboardInterrupt



In [ ]:
triples_ISCO_ESCO[["triples", "ISCO", "ESCO"]].head()